# Final R32 verification and held-out-rho summary

Every damage-efficiency value is recomputed using the actually measured held-out `rho_rel_geometry`.

In [1]:

from pathlib import Path
import csv
import json
import math
import statistics

PACKAGE = Path(r"/home/djoko.bandjur.ftnkm/Notebooks/FMLE_PE_GEOMETRY_PROBE_R32")
CONFIG = json.loads(
    (PACKAGE / "CONFIG_R32.json").read_text(encoding="utf-8")
)
OUTPUT_DIR = Path(CONFIG["output_dir"])

GROUPS = [
    ("42_123", [42, 123]),
    ("456_789", [456, 789]),
    ("1011_1213", [1011, 1213]),
]

PE_TYPES = ("learned", "sinusoidal", "rope", "alibi")

EXPECTED_DIRECTION_NAMES = (
    {"task_gradient"}
    | {f"random_{index:02d}" for index in range(32)}
)

seed_rows = []


def median(values):
    return float(statistics.median(values))


for tag, seeds in GROUPS:
    for dataset in ("imagenet", "cifar"):
        path = OUTPUT_DIR / f"{dataset}_seeds_{tag}.json"
        marker = path.with_suffix(".COMPLETE.json")

        if not path.is_file():
            raise FileNotFoundError(path)

        if not marker.is_file():
            raise FileNotFoundError(marker)

        payload = json.loads(path.read_text(encoding="utf-8"))

        for pe_type in PE_TYPES:
            for seed in seeds:
                record = payload["results"][pe_type][str(seed)]

                assert record["status"] == "ok"

                directions = record["directions"]
                assert set(directions) == EXPECTED_DIRECTION_NAMES

                task = directions["task_gradient"][
                    "heldout_geometry"
                ]

                random_names = [
                    f"random_{index:02d}"
                    for index in range(32)
                ]

                random_heldout = [
                    directions[name]["heldout_geometry"]
                    for name in random_names
                ]

                task_rho = float(task["rho_rel_geometry"])
                task_delta_ce = float(task["delta_ce"])

                # Recompute from actual held-out rho.
                task_damage_efficiency = (
                    task_delta_ce / task_rho
                )

                random_rhos = [
                    float(item["rho_rel_geometry"])
                    for item in random_heldout
                ]

                random_delta_ce = [
                    float(item["delta_ce"])
                    for item in random_heldout
                ]

                random_damage_efficiency = [
                    delta_ce / rho
                    for delta_ce, rho
                    in zip(random_delta_ce, random_rhos)
                ]

                calibration = record["direction_calibration"]

                task_gain = float(
                    calibration["task_gradient"][
                        "local_functional_gain"
                    ]
                )

                random_gains = [
                    float(
                        calibration[name][
                            "local_functional_gain"
                        ]
                    )
                    for name in random_names
                ]

                row = {
                    "dataset": dataset,
                    "pe_type": pe_type,
                    "seed": int(seed),

                    "task_rho_rel_heldout": task_rho,
                    "random_rho_rel_heldout_median":
                        median(random_rhos),

                    "task_delta_ce": task_delta_ce,
                    "random_delta_ce_median":
                        median(random_delta_ce),

                    "task_damage_efficiency_actual_rho":
                        task_damage_efficiency,

                    "random_damage_efficiency_actual_rho_median":
                        median(random_damage_efficiency),

                    "damage_efficiency_gap_task_minus_random":
                        task_damage_efficiency
                        - median(random_damage_efficiency),

                    "task_functional_gain": task_gain,

                    "random_functional_gain_median":
                        median(random_gains),

                    "task_to_random_functional_gain_ratio":
                        task_gain / max(median(random_gains), 1e-20),
                }

                for value in row.values():
                    if isinstance(value, float):
                        assert math.isfinite(value)

                seed_rows.append(row)

        print("VERIFIED:", path)


assert len(seed_rows) == 48, len(seed_rows)

seed_csv = OUTPUT_DIR / "R32_heldout_rho_seed_summary.csv"

with seed_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(seed_rows[0].keys()),
    )
    writer.writeheader()
    writer.writerows(seed_rows)


metric_names = [
    "task_rho_rel_heldout",
    "random_rho_rel_heldout_median",
    "task_delta_ce",
    "random_delta_ce_median",
    "task_damage_efficiency_actual_rho",
    "random_damage_efficiency_actual_rho_median",
    "damage_efficiency_gap_task_minus_random",
    "task_functional_gain",
    "random_functional_gain_median",
    "task_to_random_functional_gain_ratio",
]

group_summary = []

for dataset in ("imagenet", "cifar"):
    for pe_type in PE_TYPES:
        rows = [
            row for row in seed_rows
            if row["dataset"] == dataset
            and row["pe_type"] == pe_type
        ]

        assert len(rows) == 6

        summary = {
            "dataset": dataset,
            "pe_type": pe_type,
            "n_seeds": 6,
            "n_random_directions_per_seed": 32,
            "damage_efficiency_denominator":
                "actual held-out rho_rel_geometry",
        }

        for metric in metric_names:
            values = [float(row[metric]) for row in rows]

            summary[metric + "_mean"] = float(
                statistics.mean(values)
            )
            summary[metric + "_sd"] = float(
                statistics.stdev(values)
            )
            summary[metric + "_median"] = float(
                statistics.median(values)
            )

        summary[
            "positive_damage_efficiency_gap_seeds"
        ] = sum(
            row["damage_efficiency_gap_task_minus_random"] > 0
            for row in rows
        )

        group_summary.append(summary)


group_json = (
    OUTPUT_DIR / "R32_heldout_rho_group_summary.json"
)

group_json.write_text(
    json.dumps(group_summary, indent=2, allow_nan=False),
    encoding="utf-8",
)


final_marker = OUTPUT_DIR / "R32_ALL_VERIFIED.json"

final_marker.write_text(
    json.dumps({
        "status": "complete",
        "model_records": 48,
        "random_directions_per_model": 32,
        "directions_per_model": 33,
        "total_direction_evaluations": 48 * 33,
        "damage_efficiency":
            "delta_ce / actual held-out rho_rel_geometry",
        "seed_summary_csv": str(seed_csv),
        "group_summary_json": str(group_json),
    }, indent=2),
    encoding="utf-8",
)


print()
print("=" * 78)
print("ALL R32 GEOMETRY RESULTS VERIFIED")
print("Model records: 48")
print("Random directions per model: 32")
print("Directions per model: 33")
print("Total matched-direction evaluations:", 48 * 33)
print("Damage efficiency uses actual held-out rho")
print()
print("Seed summary:", seed_csv)
print("Group summary:", group_json)
print("Marker:", final_marker)
print("=" * 78)


VERIFIED: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/imagenet_seeds_42_123.json
VERIFIED: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/cifar_seeds_42_123.json
VERIFIED: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/imagenet_seeds_456_789.json
VERIFIED: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/cifar_seeds_456_789.json
VERIFIED: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/imagenet_seeds_1011_1213.json
VERIFIED: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/cifar_seeds_1011_1213.json

ALL R32 GEOMETRY RESULTS VERIFIED
Model records: 48
Random directions per model: 32
Directions per model: 33
Total matched-direction evaluations: 1584
Damage efficiency uses actual held-out rho

Seed summary: /home/djoko.bandjur.ftnkm/Notebooks/results/pe_geometry_probe_n6_r32/R32_heldout_rho_seed_summary.csv
Group summary: /home/djoko.bandjur.ftnkm/Notebooks/re